In [1]:
import pandas as pd
import os
import json
from datetime import datetime
import time
import shutil

In [2]:
# ============================================================================
# 1. Проверка и подготовка окружения
# ============================================================================
print("\n" + "="*70)
print(" ПОДГОТОВКА ОКРУЖЕНИЯ")
print("="*70)

# Путь к файлу
FILE_PATH = r"O:\extracted\mimodump-dataset.csv"

# Проверка существования файла
if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"Файл не найден: {FILE_PATH}")

# Проверка свободного места на диске
def check_disk_space():
    """Проверяет свободное место на диске и возвращает True, если достаточно места"""
    try:
        # Для Windows
        total, used, free = shutil.disk_usage("O:")
        free_gb = free / (1024**3)
        print(f"✅ Свободное место на диске O: {free_gb:.2f} ГБ")
        return free_gb > 10  # Нужно минимум 10 ГБ свободного места
    except:
        # Если не получается определить (например, на Linux)
        print(" Не удалось определить свободное место на диске")
        return True

if not check_disk_space():
    print(" НЕДОСТАТОЧНО МЕСТА НА ДИСКЕ! Завершение выполнения.")
    exit()

# Создаем директорию для результатов
OUTPUT_DIR = r"O:\extracted"
os.makedirs(OUTPUT_DIR, exist_ok=True)


 ПОДГОТОВКА ОКРУЖЕНИЯ
✅ Свободное место на диске O: 166.34 ГБ


In [3]:
# ============================================================================
# 2. Исследование структуры файла
# ============================================================================
print("\n" + "="*70)
print(" ИССЛЕДОВАНИЕ СТРУКТУРЫ ФАЙЛА")
print("="*70)

# Проверяем первые строки файла для определения структуры
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    first_lines = [f.readline().strip() for _ in range(20)]

print("\nПервые 10 строк файла:")
for i, line in enumerate(first_lines[:10], 1):
    print(f"{i:3d}: {line[:100]}{'...' if len(line) > 100 else ''}")

# Определяем разделитель и количество колонок
header_line = first_lines[0]
columns = [col.strip() for col in header_line.split(';')]
expected_cols = len(columns)

print(f"\nОбнаружено колонок в заголовке: {expected_cols}")
print(f"Колонки: {columns}")

# Проверка количества разделителей
print("\nПроверка количества ';' в первых 10 строках данных:")
for i, line in enumerate(first_lines[1:11], 2):
    count = line.count(';')
    status = "✓" if count == expected_cols - 1 else "✗"
    print(f"{i:3d}: {status} {count:2d} разделителей (ожидается {expected_cols - 1}) → {line[:60]}...")


 ИССЛЕДОВАНИЕ СТРУКТУРЫ ФАЙЛА

Первые 10 строк файла:
  1: name;price;current_price;lowest_price;msrp_price;image;item_url;category_name;category_url;available...
  2: tulup.si Slika na platnu Pokrajina desert sand;24;31.95;31.95;24;https://i.cdn.nrholding.net/5671293...
  3: ";tulup.si;tulupsi-slika-na-platnu-pokrajina-100078120788;2021-11-08 16:50:39.142181
  4: tulup.si Slika na platnu Sea coast landscape;32;34.95;34.95;32;https://i.cdn.nrholding.net/56648085;...
  5: ";tulup.si;tulupsi-slika-na-platnu-sea-coast-100073849383;2021-11-08 16:51:14.274117
  6: tulup.si Slika na platnu Slap landscape vode;35;34.95;34.95;35;https://i.cdn.nrholding.net/56635821;...
  7: ";tulup.si;tulupsi-slika-na-platnu-slap-landscape-100073848076;2021-11-08 16:51:25.548717
  8: tulup.si Slika na platnu Slap narava;24;22.95;22.95;24;https://i.cdn.nrholding.net/56695376;https://...
  9: ";tulup.si;tulupsi-slika-na-platnu-slap-narava-100073848132;2021-11-08 16:51:27.341345
 10: tulup.si Slika na platnu Sun

In [4]:
# ============================================================================
# 3. Загрузка тестовой выборки
# ============================================================================
print("\n" + "="*70)
print(" ЗАГРУЗКА ТЕСТОВОЙ ВЫБОРКИ")
print("="*70)

# Попытка загрузить небольшую выборку для проверки структуры
try:
    sample_df = pd.read_csv(
        FILE_PATH,
        sep=';',
        nrows=1000,
        on_bad_lines='skip',
        engine='python',
        encoding='utf-8',
        dtype=str,
        keep_default_na=True,
        na_values=['', 'NA', 'null', 'NULL']
    )
    
    print(f"ОК Успешно прочитано {len(sample_df):,} строк из 1000")
    print(f"Фактические колонки ({len(sample_df.columns)}):")
    for i, col in enumerate(sample_df.columns[:10], 1):
        print(f"  {i:2d}. {col}")
    if len(sample_df.columns) > 10:
        print(f"  ... и ещё {len(sample_df.columns) - 10} колонок")
    
    # Проверка наличия ключевых колонок
    key_cols = ['name', 'price', 'category_name', 'parsed_at']
    print("\nНаличие ключевых колонок:")
    for col in key_cols:
        status = "ОК" if col in sample_df.columns else "НЕОК"
        print(f"  {status} {col}")
    
except Exception as e:
    print(f"НЕОК Ошибка при чтении тестовой выборки: {type(e).__name__}: {e}")
    raise


 ЗАГРУЗКА ТЕСТОВОЙ ВЫБОРКИ
ОК Успешно прочитано 1,000 строк из 1000
Фактические колонки (21):
   1. name
   2. price
   3. current_price
   4. lowest_price
   5. msrp_price
   6. image
   7. item_url
   8. category_name
   9. category_url
  10. available
  ... и ещё 11 колонок

Наличие ключевых колонок:
  ОК name
  ОК price
  ОК category_name
  ОК parsed_at


In [1]:
# ============================================================================
# 4. Обработка данных
# ============================================================================
print("\n" + "="*70)
print(" НАЧАЛО ОБРАБОТКИ ДАННЫХ")
print("="*70)

import pandas as pd
import os
import json
from datetime import datetime
import time
import gc

# Параметры
FILE_PATH = r"O:\extracted\mimodump-dataset.csv"
OUTPUT_DIR = r"O:\extracted"
CHUNK_SIZE = 50000  # Меньше чанк = чаще прогресс
SAMPLE_FRACTION = 0.01
SAVE_EVERY = 10  # Сохраняем каждые 10 чанков

# Загружаем текущее состояние
sample_file = os.path.join(OUTPUT_DIR, "mimovrste_sample.parquet")
profile_file = os.path.join(OUTPUT_DIR, "dataset_profile.json")

if os.path.exists(sample_file) and os.path.exists(profile_file):
    print(" Загружаем текущую выборку...")
    sample_df = pd.read_parquet(sample_file)
    sample_rows = [sample_df]
    with open(profile_file, "r", encoding="utf-8") as f:
        profile = json.load(f)
    start_row = profile.get('total_rows_processed', 0)
    print(f"   Возобновляем с {start_row:,} строк")
else:
    sample_rows = []
    start_row = 0
    print(" Начинаем с нуля...")

total_rows = start_row
chunks_processed = 0
start_time = time.time()

# Обработка файла
print("\n⏳ Обработка данных (обновление каждые 50,000 строк)...")
for chunk in pd.read_csv(
    FILE_PATH,
    sep=';',
    chunksize=CHUNK_SIZE,
    dtype='object',
    on_bad_lines='skip',
    engine='python',
    encoding='utf-8'
):
    chunks_processed += 1
    total_rows += len(chunk)
    
    # Добавляем в выборку
    if len(chunk) > 0:
        sample_size = int(len(chunk) * SAMPLE_FRACTION)
        if sample_size > 0:
            sample_rows.append(chunk.sample(n=min(sample_size, len(chunk)), random_state=42))
    
    # Прогресс каждые 5 чанков
    if chunks_processed % 5 == 0:
        elapsed = time.time() - start_time
        speed = total_rows / elapsed if elapsed > 0 else 0
        print(f"Обработано: {total_rows:,} строк | Скорость: {speed:,.0f} строк/сек | Выборка: {sum(len(df) for df in sample_rows):,} строк", end='\r')
    
    # Сохранение каждые 10 чанков
    if chunks_processed % SAVE_EVERY == 0:
        # Объединяем и ограничиваем выборку
        if sample_rows:
            df = pd.concat(sample_rows, ignore_index=True)
            if len(df) > 300000:
                df = df.sample(n=300000, random_state=42)
            df.to_parquet(os.path.join(OUTPUT_DIR, "mimovrste_sample.parquet"), compression="zstd", index=False)
        
        # Сохраняем профиль
        profile = {
            "total_rows_processed": total_rows,
            "sample_size": len(df) if sample_rows else 0,
            "saved_at": datetime.now().isoformat(),
            "status": "in_progress"
        }
        with open(profile_file, "w", encoding="utf-8") as f:
            json.dump(profile, f, ensure_ascii=False, indent=2)
        
        # Очищаем память
        gc.collect()
    
    # Защита от переполнения
    if len(sample_rows) > 30:
        df = pd.concat(sample_rows, ignore_index=True).sample(n=50000, random_state=42)
        sample_rows = [df]

# Финальное сохранение
print("\n\n✅ Обработка завершена!")
if sample_rows:
    df = pd.concat(sample_rows, ignore_index=True)
    if len(df) > 200000:
        df = df.sample(n=200000, random_state=42)
    df.to_parquet(os.path.join(OUTPUT_DIR, "mimovrste_sample.parquet"), compression="zstd", index=False)

profile = {
    "total_rows_processed": total_rows,
    "sample_size": len(df) if sample_rows else 0,
    "saved_at": datetime.now().isoformat(),
    "status": "completed",
    "processing_time_sec": time.time() - start_time
}
with open(profile_file, "w", encoding="utf-8") as f:
    json.dump(profile, f, ensure_ascii=False, indent=2)

print(f"Итого: {total_rows:,} строк | Выборка: {len(df):,} строк | Время: {time.time() - start_time:.1f} сек")


 НАЧАЛО ОБРАБОТКИ ДАННЫХ
 Загружаем текущую выборку...
   Возобновляем с 357,900,000 строк

⏳ Обработка данных (обновление каждые 50,000 строк)...
Обработано: 480,900,000 строк | Скорость: 53,215 строк/сек | Выборка: 65,000 строккокк

✅ Обработка завершена!
Итого: 480,900,277 строк | Выборка: 50,002 строк | Время: 9039.1 сек


In [1]:
# ============================================================================
# ВИЗУАЛИЗАЦИЯ И АНАЛИЗ РЕЗУЛЬТАТОВ
# ============================================================================
print("\n" + "="*70)
print(" АНАЛИЗ И ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ")
print("="*70)

import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Настройки стиля
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

OUTPUT_DIR = r"O:\extracted"
VISUAL_DIR = os.path.join(OUTPUT_DIR, "visualizations")
os.makedirs(VISUAL_DIR, exist_ok=True)

# Загрузка выборки
print("Загрузка выборки для анализа...")
sample_df = pd.read_parquet(os.path.join(OUTPUT_DIR, "mimovrste_sample.parquet"))
print(f" Загружено {len(sample_df):,} строк")

# Преобразование даты (если есть колонка parsed_at)
if 'parsed_at' in sample_df.columns:
    print("Преобразование временных меток...")
    sample_df['parsed_at'] = pd.to_datetime(sample_df['parsed_at'], errors='coerce')
    sample_df = sample_df.dropna(subset=['parsed_at'])
    sample_df['year'] = sample_df['parsed_at'].dt.year
    sample_df['month'] = sample_df['parsed_at'].dt.month
    sample_df['date'] = sample_df['parsed_at'].dt.date
    print(f" Обработано дат: {len(sample_df):,}")


 АНАЛИЗ И ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
Загрузка выборки для анализа...
 Загружено 50,002 строк
Преобразование временных меток...
 Обработано дат: 50,002


In [3]:
# ============================================================================
# 5. ВИЗУАЛИЗАЦИЯ И АНАЛИЗ РЕЗУЛЬТАТОВ
# ============================================================================
print("\n" + "="*70)
print(" АНАЛИЗ И ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ")
print("="*70)

import pandas as pd
import numpy as np
import os
import json
from datetime import datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

OUTPUT_DIR = r"O:\extracted"
VISUAL_DIR = os.path.join(OUTPUT_DIR, "visualizations")
os.makedirs(VISUAL_DIR, exist_ok=True)

# Загрузка выборки
print("Загрузка выборки для анализа...")
sample_df = pd.read_parquet(os.path.join(OUTPUT_DIR, "mimovrste_sample.parquet"))
print(f" Загружено {len(sample_df):,} строк")

# ============================================================================
# 5.0. ОЧИСТКА И ПРЕОБРАЗОВАНИЕ ЦЕН В ЧИСЛОВОЙ ФОРМАТ
# ============================================================================
print("\n" + "="*70)
print(" ОЧИСТКА ДАННЫХ: ПРЕОБРАЗОВАНИЕ ЦЕН В ЧИСЛОВОЙ ФОРМАТ")
print("="*70)

# Функция для очистки цен (удаление лишних точек, замена запятых)
def clean_price(value):
    if pd.isna(value) or value == '':
        return np.nan
    # Преобразуем в строку и удаляем всё кроме цифр и одной точки
    value = str(value).strip()
    # Заменяем запятые на точки (европейский формат)
    value = value.replace(',', '.')
    # Удаляем все точки кроме последней (разделитель дробной части)
    parts = value.split('.')
    if len(parts) > 1:
        value = parts[0] + '.' + ''.join(parts[1:])
    try:
        return float(value)
    except:
        return np.nan

# Применяем очистку ко всем колонкам с ценами
price_columns = ['price', 'current_price', 'lowest_price', 'msrp_price']
for col in price_columns:
    if col in sample_df.columns:
        print(f"Очистка колонки '{col}'...")
        sample_df[col] = sample_df[col].apply(clean_price)
        valid_count = sample_df[col].notna().sum()
        print(f"  Преобразовано {valid_count:,} значений из {len(sample_df):,}")

# ============================================================================
# 5.1. АНАЛИЗ ДИНАМИКИ ЦЕН ПО КАТЕГОРИЯМ
# ============================================================================
print("\n" + "="*70)
print(" АНАЛИЗ ДИНАМИКИ ЦЕН ПО КАТЕГОРИЯМ")
print("="*70)

# Преобразование даты (если есть колонка parsed_at)
if 'parsed_at' in sample_df.columns:
    print("Преобразование временных меток...")
    sample_df['parsed_at'] = pd.to_datetime(sample_df['parsed_at'], errors='coerce')
    sample_df = sample_df.dropna(subset=['parsed_at'])
    sample_df['year'] = sample_df['parsed_at'].dt.year
    sample_df['month'] = sample_df['parsed_at'].dt.month
    sample_df['date'] = sample_df['parsed_at'].dt.date
    print(f"Обработано дат: {len(sample_df):,}")

if 'category_name' in sample_df.columns and 'price' in sample_df.columns and 'year' in sample_df.columns:
    # Фильтрация некорректных цен
    sample_df = sample_df[sample_df['price'] > 0]
    sample_df = sample_df[sample_df['price'] < 10000]  # Убираем выбросы
    
    # Топ-10 категорий по объёму данных
    top_cats = sample_df['category_name'].value_counts().head(10).index.tolist()
    cat_data = sample_df[sample_df['category_name'].isin(top_cats)]
    
    # Средние цены по годам
    price_by_year = cat_data.groupby(['year', 'category_name'])['price'].mean().reset_index()
    
    # Визуализация
    plt.figure(figsize=(14, 7))
    colors = plt.cm.tab10(range(len(top_cats)))
    for idx, cat in enumerate(top_cats):
        cat_trend = price_by_year[price_by_year['category_name'] == cat]
        if len(cat_trend) > 0:
            plt.plot(cat_trend['year'], cat_trend['price'], 
                    marker='o', label=cat[:30] + "..." if len(cat) > 30 else cat,
                    color=colors[idx % len(colors)])
    
    plt.title('Динамика средних цен по топ-10 категориям (2021-2023)', fontsize=14, fontweight='bold')
    plt.xlabel('Год', fontsize=12)
    plt.ylabel('Средняя цена (евро)', fontsize=12)
    plt.legend(title='Категории', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Сохранение
    plt.savefig(os.path.join(VISUAL_DIR, 'price_trend_categories.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print("График динамики цен сохранён: price_trend_categories.png")
    
    # Сохранение данных для отчёта
    price_by_year.to_csv(os.path.join(VISUAL_DIR, 'price_trend_categories.csv'), index=False)
    print("Данные для таблицы сохранены: price_trend_categories.csv")
    
    # Расчёт роста цен
    for cat in top_cats:
        cat_trend = price_by_year[price_by_year['category_name'] == cat]
        if len(cat_trend) >= 2:
            start_price = cat_trend.iloc[0]['price']
            end_price = cat_trend.iloc[-1]['price']
            growth = ((end_price - start_price) / start_price) * 100
            print(f"  📈 {cat[:25]:25s}: рост {growth:+.1f}% ({start_price:.2f} → {end_price:.2f} €)")
else:
    print("Недостаточно данных для анализа динамики цен")
    print(f"   Доступные колонки: {list(sample_df.columns)}")


 АНАЛИЗ И ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
Загрузка выборки для анализа...
 Загружено 50,002 строк

 ОЧИСТКА ДАННЫХ: ПРЕОБРАЗОВАНИЕ ЦЕН В ЧИСЛОВОЙ ФОРМАТ
Очистка колонки 'price'...
  Преобразовано 50,002 значений из 50,002
Очистка колонки 'current_price'...
  Преобразовано 42,582 значений из 50,002
Очистка колонки 'lowest_price'...
  Преобразовано 42,582 значений из 50,002
Очистка колонки 'msrp_price'...
  Преобразовано 44,256 значений из 50,002

 АНАЛИЗ ДИНАМИКИ ЦЕН ПО КАТЕГОРИЯМ
Преобразование временных меток...
Обработано дат: 50,002
График динамики цен сохранён: price_trend_categories.png
Данные для таблицы сохранены: price_trend_categories.csv
  📈 otroska obutev           : рост +85.5% (35.78 → 66.37 €)
  📈 erotika                  : рост +111.6% (39.78 → 84.19 €)
  📈 vsa obutev               : рост -11.5% (116.64 → 103.19 €)
  📈 modne ure                : рост +7.8% (116.07 → 125.17 €)
  📈 vrtno pohistvo           : рост -37.6% (198.80 → 124.04 €)
  📈 majice otroske           : рост +3.

In [4]:
# ============================================================================
# АНАЛИЗ СЕЗОННОСТИ ЦЕН
# ============================================================================
print("\n" + "="*70)
print(" АНАЛИЗ СЕЗОННОСТИ ЦЕН")
print("="*70)

if 'month' in sample_df.columns and 'price' in sample_df.columns:
    # Фильтрация корректных цен
    season_df = sample_df[(sample_df['price'] > 0) & (sample_df['price'] < 10000)]
    
    # Средние цены по месяцам
    seasonality = season_df.groupby('month')['price'].agg(
        mean='mean',
        median='median',
        count='count'
    ).reset_index()
    seasonality = seasonality.sort_values('month')
    
    # Визуализация
    plt.figure(figsize=(12, 6))
    sns.barplot(data=seasonality, x='month', y='mean', palette='Blues_d')
    plt.title('Сезонность цен по месяцам (2021-2023)', fontsize=14, fontweight='bold')
    plt.xlabel('Месяц', fontsize=12)
    plt.ylabel('Средняя цена (евро)', fontsize=12)
    plt.xticks(ticks=range(12), labels=['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн', 
                                       'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек'])
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    
    # Сохранение
    plt.savefig(os.path.join(VISUAL_DIR, 'seasonality.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(" График сезонности сохранён: seasonality.png")
    
    # Расчёт индексов сезонности
    avg_price = seasonality['mean'].mean()
    seasonality['seasonality_index'] = (seasonality['mean'] / avg_price * 100).round(2)
    seasonality.to_csv(os.path.join(VISUAL_DIR, 'seasonality_index.csv'), index=False)
    
    # Вывод ключевых выводов
    max_month = seasonality.loc[seasonality['mean'].idxmax(), 'month']
    min_month = seasonality.loc[seasonality['mean'].idxmin(), 'month']
    print(f" Пик цен: месяц {int(max_month)} (средняя цена {seasonality['mean'].max():.2f} €)")
    print(f" Минимум цен: месяц {int(min_month)} (средняя цена {seasonality['mean'].min():.2f} €)")
else:
    print(" Недостаточно данных для анализа сезонности")


 АНАЛИЗ СЕЗОННОСТИ ЦЕН


C:\Users\andreyb\AppData\Local\Temp\ipykernel_15996\1706310443.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=seasonality, x='month', y='mean', palette='Blues_d')


 График сезонности сохранён: seasonality.png
 Пик цен: месяц 9 (средняя цена 117.58 €)
 Минимум цен: месяц 12 (средняя цена 85.18 €)


In [5]:
# ============================================================================
# РАСЧЁТ ИНДЕКСА ПОТРЕБИТЕЛЬСКИХ ЦЕН (ИПЦ)
# ============================================================================
print("\n" + "="*70)
print(" РАСЧЁТ ИНДЕКСА ПОТРЕБИТЕЛЬСКИХ ЦЕН (ИПЦ)")
print("="*70)

if 'date' in sample_df.columns and 'price' in sample_df.columns:
    # Фильтрация корректных цен
    ipi_df = sample_df[(sample_df['price'] > 0) & (sample_df['price'] < 10000)]
    
    # Ежемесячная агрегация
    monthly_prices = ipi_df.groupby('date')['price'].mean().reset_index()
    monthly_prices = monthly_prices.sort_values('date')
    
    # Базовый период (январь 2021)
    base_period = monthly_prices[monthly_prices['date'].astype(str).str.contains('2021-01')]
    if len(base_period) > 0:
        base_price = base_period['price'].mean()
    else:
        base_price = monthly_prices['price'].iloc[0]
    
    # Расчёт индекса
    monthly_prices['price_index'] = (monthly_prices['price'] / base_price * 100).round(2)
    
    # Визуализация
    plt.figure(figsize=(14, 6))
    plt.plot(monthly_prices['date'], monthly_prices['price_index'], 
             marker='o', linewidth=2, color='#2E86AB')
    plt.axhline(y=100, color='red', linestyle='--', label='Базовый период (янв 2021)')
    plt.title('Индекс потребительских цен (ИПЦ) на Mimovrste\nБаза: январь 2021 = 100', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Дата', fontsize=12)
    plt.ylabel('Индекс цен', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Сохранение
    plt.savefig(os.path.join(VISUAL_DIR, 'price_index.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(" График ИПЦ сохранён: price_index.png")
    
    # Расчёт общего роста
    total_growth = monthly_prices['price_index'].iloc[-1] - 100
    print(f" Общий рост цен за период 2021-2023: {total_growth:+.1f}%")
    print(f"   Базовый период (янв 2021): 100.0")
    print(f"   Текущий период: {monthly_prices['price_index'].iloc[-1]:.1f}")
    
    # Сохранение данных
    monthly_prices.to_csv(os.path.join(VISUAL_DIR, 'price_index_data.csv'), index=False)
else:
    print(" Недостаточно данных для расчёта ИПЦ")


 РАСЧЁТ ИНДЕКСА ПОТРЕБИТЕЛЬСКИХ ЦЕН (ИПЦ)
 График ИПЦ сохранён: price_index.png
 Общий рост цен за период 2021-2023: -69.1%
   Базовый период (янв 2021): 100.0
   Текущий период: 30.9


In [3]:
# ============================================================================
# 5.4. АНАЛИЗ ВОЛАТИЛЬНОСТИ ЦЕН ПО КАТЕГОРИЯМ
# ============================================================================
print("\n" + "="*70)
print(" АНАЛИЗ ВОЛАТИЛЬНОСТИ ЦЕН ПО КАТЕГОРИЯМ")
print("="*70)

import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

OUTPUT_DIR = r"O:\extracted"
VISUAL_DIR = os.path.join(OUTPUT_DIR, "visualizations")
os.makedirs(VISUAL_DIR, exist_ok=True)

# ============================================================================
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# ============================================================================
print("Загрузка данных для анализа...")
sample_df = pd.read_parquet(os.path.join(OUTPUT_DIR, "mimovrste_sample.parquet"))
print(f" Загружено {len(sample_df):,} строк")

# Функция очистки цен
def clean_price(value):
    if pd.isna(value) or value == '':
        return np.nan
    value = str(value).strip().replace(',', '.')
    parts = value.split('.')
    if len(parts) > 1:
        value = parts[0] + '.' + ''.join(parts[1:])
    try:
        return float(value)
    except:
        return np.nan

# Очистка цен
if 'price' in sample_df.columns and sample_df['price'].dtype == 'object':
    print("Очистка цен...")
    sample_df['price'] = sample_df['price'].apply(clean_price)
    print(" Цены очищены")

# Фильтрация корректных цен
sample_df = sample_df[(sample_df['price'] > 0) & (sample_df['price'] < 10000)]

# ============================================================================
# АНАЛИЗ ВОЛАТИЛЬНОСТИ
# ============================================================================
if 'category_name' in sample_df.columns and 'price' in sample_df.columns:
    # Расчёт коэффициента вариации
    volatility = sample_df.groupby('category_name')['price'].agg(
        mean='mean',
        std='std',
        count='count'
    ).reset_index()
    volatility = volatility[volatility['count'] >= 20]  # Только категории с достаточным объёмом
    volatility['cv'] = (volatility['std'] / volatility['mean'] * 100).round(2)
    volatility = volatility.sort_values('cv', ascending=False).head(15)
    
    # Визуализация
    plt.figure(figsize=(14, 7))
    sns.barplot(data=volatility, x='cv', y='category_name', palette='Reds_r')
    plt.title('Топ-15 категорий по волатильности цен\n(коэффициент вариации)', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Коэффициент вариации, %', fontsize=12)
    plt.ylabel('Категория', fontsize=12)
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    
    # Сохранение
    plt.savefig(os.path.join(VISUAL_DIR, 'volatility.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(" График волатильности сохранён: volatility.png")
    
    # Вывод ключевых выводов
    most_volatile = volatility.iloc[0]
    most_stable = volatility.iloc[-1]
    print(f" Самая волатильная категория: {most_volatile['category_name'][:40]} ({most_volatile['cv']:.1f}% CV)")
    print(f"  Самая стабильная категория: {most_stable['category_name'][:40]} ({most_stable['cv']:.1f}% CV)")
    
    # Сохранение данных
    volatility.to_csv(os.path.join(VISUAL_DIR, 'volatility_data.csv'), index=False)
    print(" Данные волатильности сохранены: volatility_data.csv")
else:
    print(" Недостаточно данных для анализа волатильности")
    print(f"   Доступные колонки: {list(sample_df.columns)}")


 АНАЛИЗ ВОЛАТИЛЬНОСТИ ЦЕН ПО КАТЕГОРИЯМ
Загрузка данных для анализа...
 Загружено 50,002 строк
Очистка цен...
 Цены очищены


C:\Users\andreyb\AppData\Local\Temp\ipykernel_4984\445115469.py:69: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=volatility, x='cv', y='category_name', palette='Reds_r')


 График волатильности сохранён: volatility.png
 Самая волатильная категория: vzglavniki (385.0% CV)
  Самая стабильная категория: moske sportne majice in topi (200.7% CV)
 Данные волатильности сохранены: volatility_data.csv


In [4]:
# ============================================================================
# ЗАДАЧА 1: АНАЛИЗ РАСПРЕДЕЛЕНИЯ ЦЕН (статистическая)
# ============================================================================
print("\n Задача 1: Анализ распределения цен по категориям")
if 'category_name' in sample_df.columns and 'price' in sample_df.columns:
    # Топ-5 категорий по объёму
    top_cats = sample_df['category_name'].value_counts().head(5).index.tolist()
    
    plt.figure(figsize=(14, 8))
    for i, cat in enumerate(top_cats, 1):
        cat_data = sample_df[sample_df['category_name'] == cat]['price']
        plt.subplot(2, 3, i)
        sns.histplot(cat_data, bins=30, kde=True, color='skyblue')
        plt.title(f'{cat[:20]}...', fontsize=10)
        plt.xlabel('Цена (евро)')
        plt.ylabel('Частота')
    
    plt.tight_layout()
    plt.savefig(os.path.join(VISUAL_DIR, 'price_distribution.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(" Сохранено: price_distribution.png")
    
    # Статистика распределения
    stats = sample_df.groupby('category_name')['price'].agg(
        median='median',
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        min='min',
        max='max'
    ).reset_index()
    stats['iqr'] = stats['q75'] - stats['q25']
    stats.to_csv(os.path.join(VISUAL_DIR, 'price_distribution_stats.csv'), index=False)
    print(" Сохранена статистика распределения")


 Задача 1: Анализ распределения цен по категориям
 Сохранено: price_distribution.png
 Сохранена статистика распределения


In [5]:
# ============================================================================
# ЗАДАЧА 2: СТРУКТУРА АССОРТИМЕНТА (статистическая)
# ============================================================================
print("\n Задача 2: Анализ структуры ассортимента по категориям")
if 'category_name' in sample_df.columns:
    category_share = sample_df['category_name'].value_counts(normalize=True).head(10) * 100
    
    plt.figure(figsize=(10, 8))
    plt.pie(category_share.values, labels=category_share.index.str[:25], autopct='%1.1f%%', startangle=90)
    plt.title('Структура ассортимента: доля топ-10 категорий', fontsize=14, fontweight='bold')
    plt.axis('equal')
    plt.tight_layout()
    plt.savefig(os.path.join(VISUAL_DIR, 'assortment_structure.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(" Сохранено: assortment_structure.png")
    print(f"   Топ-3 категории занимают {category_share[:3].sum():.1f}% ассортимента")


 Задача 2: Анализ структуры ассортимента по категориям
 Сохранено: assortment_structure.png
   Топ-3 категории занимают 6.7% ассортимента


In [6]:
# ============================================================================
# ЗАДАЧА 3: ВЫЯВЛЕНИЕ АНОМАЛЬНЫХ СКАЧКОВ ЦЕН (исследовательская)
# ============================================================================
print("\n Задача 3: Выявление аномальных скачков цен в 2022 году")
if 'parsed_at' in sample_df.columns and 'price' in sample_df.columns:
    sample_df['parsed_at'] = pd.to_datetime(sample_df['parsed_at'], errors='coerce')
    sample_df = sample_df.dropna(subset=['parsed_at'])
    sample_df['month'] = sample_df['parsed_at'].dt.to_period('M')
    
    # Средние цены по месяцам
    monthly_avg = sample_df.groupby('month')['price'].mean().sort_index()
    monthly_std = sample_df.groupby('month')['price'].std().sort_index()
    
    # Расчёт аномалий (отклонение > 2 стандартных отклонения)
    anomalies = []
    for i in range(2, len(monthly_avg)):
        prev_avg = monthly_avg.iloc[i-2:i].mean()
        prev_std = monthly_std.iloc[i-2:i].mean()
        current = monthly_avg.iloc[i]
        if abs(current - prev_avg) > 2 * prev_std:
            anomalies.append((monthly_avg.index[i], current, prev_avg))
    
    if anomalies:
        print(f"  Обнаружено {len(anomalies)} аномальных скачков цен:")
        for month, current, prev in anomalies[:3]:
            change = ((current - prev) / prev) * 100
            print(f"   • {month}: {change:+.1f}% (с {prev:.2f} до {current:.2f} €)")
    else:
        print(" Аномальных скачков не обнаружено")
    
    # Визуализация
    plt.figure(figsize=(14, 6))
    plt.plot(monthly_avg.index.astype(str), monthly_avg.values, marker='o', linewidth=2)
    plt.title('Средние цены по месяцам с выделением аномалий', fontsize=14, fontweight='bold')
    plt.xlabel('Месяц')
    plt.ylabel('Средняя цена (евро)')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(VISUAL_DIR, 'price_anomalies.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(" Сохранено: price_anomalies.png")


 Задача 3: Выявление аномальных скачков цен в 2022 году
 Аномальных скачков не обнаружено
 Сохранено: price_anomalies.png


In [7]:
# ============================================================================
# ЗАДАЧА 4: АНАЛИЗ ЦЕН ПО БРЕНДАМ (статистическая)
# ============================================================================
print("\n Задача 4: Анализ цен по брендам")
if 'brand_name' in sample_df.columns and 'price' in sample_df.columns:
    # Топ-10 брендов по количеству товаров
    top_brands = sample_df['brand_name'].value_counts().head(10).index.tolist()
    brand_stats = sample_df[sample_df['brand_name'].isin(top_brands)].groupby('brand_name')['price'].agg(
        median='median',
        mean='mean',
        count='count'
    ).sort_values('median', ascending=False)
    
    plt.figure(figsize=(12, 7))
    sns.barplot(data=brand_stats.reset_index(), x='median', y='brand_name', palette='viridis')
    plt.title('Медианные цены по топ-10 брендам', fontsize=14, fontweight='bold')
    plt.xlabel('Медианная цена (евро)')
    plt.ylabel('Бренд')
    plt.tight_layout()
    plt.savefig(os.path.join(VISUAL_DIR, 'brand_prices.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(" Сохранено: brand_prices.png")
    print(f"   Самый дорогой бренд: {brand_stats.index[0]} ({brand_stats.iloc[0]['median']:.2f} €)")
    print(f"   Самый дешёвый бренд: {brand_stats.index[-1]} ({brand_stats.iloc[-1]['median']:.2f} €)")



 Задача 4: Анализ цен по брендам


C:\Users\andreyb\AppData\Local\Temp\ipykernel_4984\2510245586.py:15: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=brand_stats.reset_index(), x='median', y='brand_name', palette='viridis')


 Сохранено: brand_prices.png
   Самый дорогой бренд: shumee (124.00 €)
   Самый дешёвый бренд: Gap (20.34 €)


In [8]:
# ============================================================================
# ЗАДАЧА 5: ВЫЯВЛЕНИЕ ВЫБРОСОВ (исследовательская)
# ============================================================================
print("\n Задача 5: Выявление выбросов в ценах (метод межквартильного размаха)")
if 'price' in sample_df.columns:
    Q1 = sample_df['price'].quantile(0.25)
    Q3 = sample_df['price'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = sample_df[(sample_df['price'] < lower_bound) | (sample_df['price'] > upper_bound)]
    outlier_pct = (len(outliers) / len(sample_df)) * 100
    
    print(f"   Нижняя граница: {lower_bound:.2f} €")
    print(f"   Верхняя граница: {upper_bound:.2f} €")
    print(f"   Выбросов обнаружено: {len(outliers):,} ({outlier_pct:.2f}%)")
    
    # Визуализация выбросов
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=sample_df, y='price', width=0.5)
    plt.title('Распределение цен с выделением выбросов', fontsize=14, fontweight='bold')
    plt.ylabel('Цена (евро)')
    plt.yscale('log')  # Логарифмическая шкала для лучшей визуализации
    plt.tight_layout()
    plt.savefig(os.path.join(VISUAL_DIR, 'price_outliers.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(" Сохранено: price_outliers.png")


 Задача 5: Выявление выбросов в ценах (метод межквартильного размаха)
   Нижняя граница: -92.95 €
   Верхняя граница: 194.89 €
   Выбросов обнаружено: 5,906 (11.81%)
 Сохранено: price_outliers.png
